In [0]:
Suppose in future you want to debug Option 2 is always better choice. In real work actual query will be 100 times bigger than this. When you do not write cleanly it is extremely difficult for others after few months what exactly it is written.

            --1 Write clean and formatted SQL --
            --(Uppercase: SELECT, FORM, JOIN. Lowercse: column name, table name, alias) 
-- Option 1
select o.order_id ,o.order_date,c.customerid,c.fullName,c.email
from orders o join customers c on o.customerid=c.customerid
where o.order_date>='2025-01-01' and o.order_date<'2025-01-02' and c.status='ACTIVE'
order by o.order_date desc,o.order_id;


-- Option 2 ✅ 
SELECT
    o.order_id,
    o.order_date,
    c.customer_id,
    c.full_name,
    c.email
FROM orders AS o
JOIN customers AS c
    ON o.customer_id = c.customer_id
WHERE o.order_date >= DATE '2025-01-01'
  AND o.order_date < DATE '2025-01-02'
  AND c.status = 'ACTIVE'
ORDER BY
    o.order_date DESC,
    o.order_id;
-----------------------------
                --  2 Fetch only what you need

-- Option 1
SELECT *
FROM orders
WHERE order_date >= DATE '2025-01-01';


-- Option 2 ✅ 
SELECT
    order_id,
    customer_id,
    order_date,
    total_amount
FROM orders
WHERE order_date >= DATE '2025-01-01'
LIMIT 100;

-- Advantage: (1) If you are doing some analysis what the data in the table always use limit. This will save lot of costs specially cloud bassed data analysis & improve the performance instead of fetching million of records. (2) * vs Specific column: Internally what database is doing for each column it goes to the disk where the data is stored. It reads the data and return the data to the network. This is IO operation. it needs to do for 100 of columns. when you putting star same thing have to do for 100 of columns. (3) Suppose you have written * for specific query which can handle 100s of record now after some day 4 extra column added on that time the sql query will breaks but for specific column it will work fine.
-----------------------------
                -- 3 Use CTE's to break down complex logic

-- Option 1 (Multiple nested subquery)
SELECT
    AVG(avg_order_value) AS avg_order_value_in_busy_months
FROM (
    SELECT
        month_start,
        total_orders,
        total_revenue,
        total_revenue / NULLIF(total_orders, 0) AS avg_order_value
    FROM (
        SELECT
            DATE_TRUNC('month', order_date) AS month_start,
            COUNT(*) AS total_orders,
            SUM(total_amount) AS total_revenue
        FROM orders
        WHERE order_date >= DATE '2025-01-01'
          AND order_date <  DATE '2026-01-01'
        GROUP BY DATE_TRUNC('month', order_date)
    ) t1
) t2
WHERE total_orders > 1000;


-- Option 2 (Using CTE for redability, performance)  ✅ 
WITH monthly_orders AS (
    SELECT
        DATE_TRUNC('month', order_date) AS month_start,
        COUNT(*) AS total_orders,
        SUM(total_amount) AS total_revenue
    FROM orders
    WHERE order_date >= DATE '2025-01-01'
      AND order_date <  DATE '2026-01-01'
    GROUP BY DATE_TRUNC('month', order_date)
),
high_volume_months AS ( --2nd subquery/2nd cte
    SELECT
        month_start,
        total_orders,
        total_revenue,
        total_revenue / NULLIF(total_orders, 0) AS avg_order_value
    FROM monthly_orders
    WHERE total_orders > 1000
)
SELECT
    AVG(avg_order_value) AS avg_order_value_in_busy_months
FROM high_volume_months;  
-----------------------------
            -- 4 Prefer explicit JOINs over "everything in a subquery"           
-- Version 1
SELECT
    o.order_id,
    o.order_date,
    (SELECT c.full_name
     FROM customers c
     WHERE c.customer_id = o.customer_id) AS customer_name
FROM orders o;


-- Version 2
SELECT
    o.order_id,
    o.order_date,
    c.full_name AS customer_name
FROM orders AS o
JOIN customers AS c
    ON o.customer_id = c.customer_id;
-----------------------------
            -- 5 Write index-friendly predicates
--- order_date is DATE & index exists on it ---
If you have a column which you have an index and you want to use it in where clause, then make sure you wre not using that column within the function. Because otherwise SQL optimizer will not be able to use the index. this could happen because index is created on the values stored in the column, and when you enclose this column within the function it values could changed and index could not be used.

-- Version 1 (used entire table scan)
SELECT
    order_id,
    customer_id,
    order_date
FROM orders
WHERE DATE(order_date) = DATE '2025-01-01';


-- Version 2  (High probablity index will be used )
SELECT
    order_id,
    customer_id,
    order_date
FROM orders
WHERE order_date = DATE '2025-01-01';       
-----------------------------
                -- 6 Use DISTINCT correctly 
Distinct fetch unique value. it should not be used to cover up the bad data returned from query. EX: you have query and returning some duplicate data. Because you did some wrong Join. But the dataset doesnot ahve duplicate. To fix it you are using distinct. But final query works but it is not right approach. You should fixed underline join rather than using distinct.         

EX: Fetching those customers who has made some orders.
-- Version 1
SELECT DISTINCT
    c.customer_id,
    c.full_name
FROM customers c
JOIN orders o
    ON c.customer_id = o.customer_id;

-- Version 2  This way I will always get only one record for each customer
SELECT
    c.customer_id,
    c.full_name
FROM customers c
WHERE EXISTS (
    SELECT 1
    FROM orders o
    WHERE o.customer_id = c.customer_id
);
-----------------------------
            -- COUNT(*) vs COUNT(column)
count(*) always return total no of rows irrespective of whether there is null value or not in any of the column.
count(column) return total no of non-null value in that specific column. 

-- Query 1 total no of user present in table,
SELECT COUNT(*) AS total_users
FROM users;


-- Query 2 return total no of user where "last_login_at" column is not null. EX: if we have 100 users and 30 users fild is null. So it will retun 70
SELECT COUNT(last_login_at) AS users_with_login
FROM users;

